# GUI 1: Chat Interface

Running an agent via `async for event in agent.run(...)` in a notebook is convenient for demos, but interacting with a coding agent naturally — typing requests, watching tool calls execute live, and reading streaming responses — calls for a proper interface. This notebook wraps the complete agent stack from the previous six notebooks in a Flet desktop application.

The result is a dark-themed chat window: a scrollable feed of message bubbles and tool-call cards in the center, a text input bar at the bottom, and a thin status bar at the top showing model, turn count, and token usage. Every `AgentEvent` emitted by `agent.run(...)` maps to a concrete UI mutation — a new bubble, an updated card, a refreshed counter. The agent runs in an async background task so the interface stays responsive throughout long multi-turn sessions.

## Application Architecture

The UI package lives at `src/notebooks/agent/ui/` and contains two files. `components.py` defines the reusable display widgets and their data models — everything needed to render a single conversation item. `app.py` defines the `AgentApp` page controller, which wires Flet's event loop to the agent's async generator.

```
AgentApp (page controller)
├── Agent (agentic loop)
│   └── Session → LLMClient + ToolRegistry
├── ApprovalManager (approval decisions)
└── Flet Page
    ├── StatusBar (model · turn · tokens)
    ├── ListView (message feed)
    │   ├── MessageBubble (user / assistant / system)
    │   └── ToolCallCard (tool invocations)
    └── InputBar (TextField + Send button)
```

`AgentApp` holds five pieces of mutable state: the Flet `page`, the `Agent` instance, an `ApprovalManager`, a `_pending_cards` dict mapping in-flight call IDs to their live card controls, and three counters (`_turn`, `_total_tokens`, `_running`). The separation between `components.py` and `app.py` keeps rendering logic stateless and independently testable — each factory function is a pure function from a data class to a Flet control.

Importing the `ui` package and listing its public names:

In [ ]:
import notebooks.agent.ui as ui

print(ui.__all__)

## UI Components

All components are defined in `components.py`. Each factory function accepts a data class and returns a `ft.Container` (or `ft.AlertDialog`) with no side effects on the page. The calling code in `app.py` is responsible for appending the control to the feed and calling `update()`.

### MessageBubble

`make_message_bubble(msg: MessageData)` renders a single conversation turn. The alignment and background color depend on `msg.role`: user messages sit on the right with a blue (`#1565C0`) background; assistant messages sit on the left with a dark-grey (`#2D2D2D`) background; system messages are centered, italic, and muted. All non-system bubbles contain a `selectable=True` `ft.Text` so users can copy output with the mouse.

**MessageBubble.** Creating one bubble of each role:

In [ ]:
from notebooks.agent.ui.components import MessageData, make_message_bubble

roles = ["user", "assistant", "system"]
msgs  = [MessageData(role=r, content=f"Hello from {r}") for r in roles]

for msg in msgs:
    bubble = make_message_bubble(msg)
    align  = bubble.alignment
    bg     = bubble.bgcolor or "(none — system)"
    print(f"{msg.role:10s}  bgcolor={bg!s:20s}  alignment=({align.x}, {align.y})")

### ToolCallCard

`make_tool_call_card(tc: ToolCallData)` renders a compact dark card for a single tool invocation. The status indicator in the top-left changes with `tc.success`: `None` (in-progress) shows a `ft.ProgressRing` spinner; `True` shows a green `CHECK_CIRCLE` icon; `False` shows a red `ERROR` icon. Below the header row, arguments are formatted as a single truncated line and the output (or error) is shown in a small monospace text block.

**ToolCallCard.** Creating an in-progress card and a succeeded card:

In [ ]:
import flet as ft
from notebooks.agent.ui.components import ToolCallData, make_tool_call_card

tc_pending = ToolCallData(
    call_id="c1",
    name="read_file",
    arguments={"path": "src/main.py"},
)
tc_done = ToolCallData(
    call_id="c2",
    name="shell",
    arguments={"command": "pytest tests/"},
    output="5 passed in 0.42s",
    success=True,
)

for tc in [tc_pending, tc_done]:
    card = make_tool_call_card(tc)
    header_row = card.content.controls[0]       # ft.Row
    status_ctrl = header_row.controls[0]        # spinner or icon
    status_type = type(status_ctrl).__name__
    print(f"{tc.name:12s}  success={str(tc.success):5s}  status_widget={status_type}")

### StatusBar

`make_status_bar(turn, total_tokens, model, running)` returns a thin `ft.Container` fixed to the top of the page. It renders model name, turn count, and token count separated by `·` glyphs. When `running=True`, a small `ft.ProgressRing` is appended to the right of the row to signal that the agent is active.

**StatusBar.** Inspecting the control row with and without the running spinner:

In [ ]:
from notebooks.agent.ui.components import make_status_bar

bar_idle    = make_status_bar(turn=3, total_tokens=12_480, model="anthropic/claude-sonnet-4")
bar_running = make_status_bar(turn=3, total_tokens=12_480, model="anthropic/claude-sonnet-4", running=True)

idle_parts    = bar_idle.content.controls
running_parts = bar_running.content.controls

print(f"Idle    controls : {len(idle_parts)}  ({', '.join(type(c).__name__ for c in idle_parts)})")
print(f"Running controls : {len(running_parts)}  ({', '.join(type(c).__name__ for c in running_parts)})")

### ApprovalDialog

`make_approval_dialog(tool_name, reason, on_approve, on_reject)` builds a modal `ft.AlertDialog` that pauses the agent loop until the user responds. The dialog title is fixed to `"Approve tool call?"`. Two action buttons — `FilledButton("Approve")` and `TextButton("Reject")` — call the provided callbacks, which resolve a `Future` that `_run_agent` is awaiting. `AgentApp` instantiates an `ApprovalManager` with the default `ON_REQUEST` policy, but wiring it into `_run_agent` is left as an extension exercise — the current implementation always shows the dialog when an approval callback fires.

**ApprovalDialog.** Constructing a dialog and inspecting its title and action count:

In [ ]:
from notebooks.agent.ui.components import make_approval_dialog

dlg = make_approval_dialog(
    tool_name="shell",
    reason="Executing: rm -rf /tmp/scratch",
    on_approve=lambda e: print("approved"),
    on_reject=lambda e: print("rejected"),
)

print(f"Title   : {dlg.title.value!r}")
print(f"Actions : {len(dlg.actions)} buttons")
for btn in dlg.actions:
    print(f"  {type(btn).__name__}: {btn.text!r}")

## The Application Controller

`AgentApp` in `app.py` is the imperative page controller. Its `__init__` runs four steps in order: configure the page, build the config and agent, initialise the five pieces of mutable UI state, and build the layout. All subsequent interaction happens through `_on_send` (triggered by the send button or Enter) and `_run_agent` (the async event-dispatch loop).

### Page Configuration and Layout

`_configure_page()` sets the window title to `"Coding Agent"`, the background color to `#121212`, and fixes the initial window size to $900 \times 700$ pixels with a minimum of $480 \times 400$.

`_build_layout()` constructs a single full-height `ft.Column` with three children: the `_status_row` at the top (a `ft.Row` wrapping the status bar container), an expanded `ft.Container` holding the `_feed` `ListView`, and the `input_bar` at the bottom. All three share `spacing=0` so there are no gaps.

**Page configuration.** Inspecting `_configure_page` and `_build_layout` source:

In [ ]:
import inspect
from notebooks.agent.ui.app import AgentApp

print(inspect.getsource(AgentApp._configure_page))
print("---")
print(inspect.getsource(AgentApp._build_layout))

### Sending a Message

`_on_send` is the entry point for every user interaction. It (1) strips the input text and returns early if empty or if the agent is already running, (2) clears the text field and appends a user bubble immediately so the UI feels responsive, (3) sets `_running = True` and disables the send button, (4) calls `_refresh_status(running=True)` to show the spinner, and (5) awaits `_run_agent(text)`. When the agent finishes, `_on_send` reverses all the disabled states and calls `_refresh_status(running=False)`.

**Sending a message.** Inspecting `_on_send` source:

In [ ]:
print(inspect.getsource(AgentApp._on_send))

### Running the Agent

`_run_agent(user_message)` iterates over `agent.run(user_message)` and dispatches each `AgentEvent` to a UI mutation:

- `TEXT_DELTA` — on the first chunk, `_add_message` creates an assistant bubble and stores a reference in `text_bubble_ref`; subsequent chunks update `bubble.content.value` in-place and call `bubble.update()`.
- `TEXT_COMPLETE` — clears `current_text` and `text_bubble_ref` so the next response starts fresh.
- `TOOL_CALL_START` — `_add_tool_card` appends an in-progress card to the feed and stores it in `_pending_cards[call_id]`.
- `TOOL_CALL_COMPLETE` — pops the pending card from `_pending_cards` and calls `_update_tool_card`, which replaces the card in-place at the same list index.
- `AGENT_END` — increments `_turn` and accumulates `total_tokens` from the usage dict.
- `AGENT_ERROR` — appends a system-role message bubble with the error text prefixed by `⚠`.

**Running the agent.** Inspecting `_run_agent` source:

In [ ]:
print(inspect.getsource(AgentApp._run_agent))

### Feed Helpers

Three short methods manage the feed list. `_add_message` calls `make_message_bubble`, appends the result to `_feed.controls`, and calls `_feed.update()`. `_add_tool_card` does the same via `make_tool_call_card`. `_update_tool_card` locates the old card by index, replaces it with a new card built from the completed `ToolCallData`, and updates the feed.

**Feed helpers.** Inspecting `_add_message`, `_add_tool_card`, and `_update_tool_card`:

In [ ]:
for method in [AgentApp._add_message, AgentApp._add_tool_card, AgentApp._update_tool_card]:
    print(inspect.getsource(method))
    print("---")

## Launching the App

The application is launched with:

In [ ]:
# uv run flet run src/notebooks/agent/ui/app.py

`OPENROUTER_API_KEY` must be set in the shell environment before running the command. The agent will refuse to start and raise a configuration error otherwise.

Below is a screenshot captured from a live session:

![**Figure.** Coding Agent desktop UI — a multi-turn session showing a user message, streaming assistant text, and two completed tool-call cards.](./img/05-app.png)

:::{.callout-note}
The approval system is wired but the `ON_REQUEST` policy requires a live UI hook to display the `ApprovalDialog` and resolve the `Future`. For unattended testing, set `approval=ApprovalPolicy.YOLO` in `_build_config()` — this bypasses the approval gate entirely and lets every tool call proceed without confirmation.

:::

## Summary

| Component | File | Role |
|---|---|---|
| `MessageData` / `ToolCallData` | `ui/components.py` | Data classes for rendering chat messages and tool invocations |
| `make_message_bubble` | `ui/components.py` | Chat bubble renderer; role-aware alignment and color |
| `make_tool_call_card` | `ui/components.py` | Tool invocation card with live status indicator |
| `make_status_bar` | `ui/components.py` | Thin header showing model, turn count, and token usage |
| `make_approval_dialog` | `ui/components.py` | Modal dialog for approving or rejecting tool calls |
| `AgentApp` | `ui/app.py` | Page controller — wires Flet events to the agent's async generator |

: {tbl-colwidths="[28,22,50]"}

<br>

This is the first of three UI notebooks. The next two add extra tools and persistence, then MCP integration. Each layer remains a self-contained module that can be used or replaced independently.

← [Hooks System](/notebooks/apps/cda/06-hooks.html) &emsp; → [GUI 2: Tools & Persistence](/notebooks/apps/cda/08-ui2.html)

---

■